# Ollama Model Server (Google Colab)

Runs [Ollama](https://ollama.com) inside this Colab VM and exposes it over a public
HTTPS tunnel so the Strands agents in this repo can point `STRANDS_MODEL_PROVIDER=ollama`
at a free GPU/CPU backend instead of a local machine.

**How this fits the project:** once you have a public URL from this notebook, set it as
`OLLAMA_HOST` in your local `.env` (see `.env.example`) and the agents in `src/agents/model_provider.py`
will call this Colab-hosted model instead of `http://localhost:11434`.

> ⚠️ **Security note:** the tunnel URL is public and has no authentication. Anyone with
> the URL can send prompts to your model. Only use this for short-lived demos/hackathons,
> never for real or sensitive data, and stop the tunnel/runtime when you're done.
> Colab free-tier sessions also disconnect after a period of inactivity or after ~12 hours,
> so the URL is temporary and will need to be regenerated each session.

## 1. Install Ollama

In [8]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 2. Start the Ollama server in the background

Colab notebook cells run in the foreground, so `ollama serve` is launched as a
background process and we poll the local API until it responds.

In [9]:
import os
import subprocess
import time

import requests

# Colab has no systemd, so run the server as a detached background process.
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
server_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for attempt in range(30):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2)
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start in time - check the logs and retry.")

Ollama server is up.


## 3. Pull a model

Pick a model that fits Colab's free-tier resources. Smaller instruction-tuned
models (e.g. `llama3.2:3b`, `phi3:mini`, `qwen2.5:3b`) work best without a paid GPU tier.
Change `MODEL_NAME` below to whatever you want the Strands agents to use.

In [10]:
MODEL_NAME = "llama3.2:3b"  # keep in sync with OLLAMA_MODEL_ID in your local .env

!ollama pull {MODEL_NAME}

In [11]:
VISION_MODEL_NAME = "minicpm-v"  # keep in sync with OLLAMA_VISION_MODEL in your local .env

!ollama pull {VISION_MODEL_NAME}

### 3b. (Optional) Pull a vision model for OCR/image extraction

Needed if you want `src/ingestion/document_reader.py` to OCR images, scanned
PDF pages, or images embedded in DOCX/PPTX files. Recommended options:

- `minicpm-v` (default, ~8B) - strong OCR on dense document scans/photos.
- `moondream` - fastest/lightest, weaker at reading dense/small text.
- `llama3.2-vision` (~11B) - strongest general-purpose vision-language model, needs more memory.

Keep this in sync with `OLLAMA_VISION_MODEL` in your local `.env`.

## 4. Quick local sanity check

Confirms the model answers before exposing it publicly.

In [12]:
response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={"model": MODEL_NAME, "prompt": "Say hello in one short sentence.", "stream": False},
    timeout=60,
)
response.raise_for_status()
print(response.json()["response"])

Hello.


## 5. Expose the server publicly with a Cloudflare tunnel

[`cloudflared`](https://github.com/cloudflare/cloudflared) needs no account/signup for
ephemeral "quick tunnels", making it the simplest way to get a public HTTPS URL pointing
at port `11434` here. The tunnel and Colab runtime are both temporary - re-run this
section any time you get a new URL.

In [13]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import re
import subprocess
import time

tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 30
while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Could not find the tunnel URL yet - re-run this cell or check the logs.")

print(f"Public Ollama endpoint: {public_url}")

cloudflared: Text file busy
Public Ollama endpoint: https://rural-soma-demonstrates-interfaces.trycloudflare.com


## 6. Use this endpoint from the local project

In your local `.env` (copy from `.env.example` if you haven't already), set:

```
STRANDS_MODEL_PROVIDER=ollama
OLLAMA_HOST=<public_url printed above>
OLLAMA_MODEL_ID=llama3.2:3b
OLLAMA_VISION_MODEL=minicpm-v
```

`src/agents/model_provider.py` reads the text-model vars to build the `OllamaModel`
used by every agent in `src/agents/`, and `src/ingestion/vision.py` reads
`OLLAMA_HOST`/`OLLAMA_VISION_MODEL` to OCR images, scanned PDF pages, and images
embedded in DOCX/PPTX submissions - so no other code changes are needed. Just keep
this Colab notebook's runtime alive while you use the app, and update `.env` again
whenever you restart this notebook (the tunnel URL changes each time).

Keep this notebook's runtime running for the duration of your demo; stopping/disconnecting
the runtime shuts down both Ollama and the tunnel.

## 7. Shutdown (run when you're done)

Stops the tunnel and the Ollama server so nothing keeps running in the background.

In [ ]:
tunnel_process.terminate()
server_process.terminate()
print("Tunnel and Ollama server stopped.")